In [1]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
import pyspark.sql.functions as F
import numpy as np
import datetime as dt
import random as rd
import subprocess
from xml.etree import ElementTree as ET

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

In [2]:
START_DATE : str = "2025-01-01"						# Simulation start date
START_TIME : str = "06:00:00"						# Simulation start time
NB_DAYS : int = 5									# Number of days to simulate
EDGES_MAX_SPEED : float = 27.78						# Max speed of the edges in m/s (120 km/h)
TRAIN_SPEED : float = 33.33							# Train speed in m/s (120 km/h)
DIRECTORY : str = "sumo_data"
OUTPUT_DIRECTORY : str = "new_start"
START_DATETIME : dt.datetime = dt.datetime.strptime(f"{START_DATE} {START_TIME}", "%Y-%m-%d %H:%M:%S")
END_DATETIME : dt.datetime = START_DATETIME + dt.timedelta(days=NB_DAYS)

filenames = {
	"punctuality" : f"{DIRECTORY}/punctuality/202501.csv",
	"trains" : f"{DIRECTORY}/trains.add.xml",
    "tracks_xml" : f"{OUTPUT_DIRECTORY}/tracks.edg.xml",
    "tracks_csv" : f"{DIRECTORY}/main_tracks.csv",
	"network" : f"{OUTPUT_DIRECTORY}/network.net.xml",
	"platforms_xml" : f"{OUTPUT_DIRECTORY}/platforms.add.xml",
	"platforms_csv" : "station_track_assigned.csv",
    "station_to_station" : f"{DIRECTORY}/station_to_station.csv"
}

output = {
	"routes" : f"{OUTPUT_DIRECTORY}/routes.rou.xml",
	"schedule" : f"{OUTPUT_DIRECTORY}/schedule.trips.xml",
	"weight_src" : f"{OUTPUT_DIRECTORY}/weights.src.xml",
	"weight_dst" : f"{OUTPUT_DIRECTORY}/weights.dst.xml",
}

In [3]:
spark : sql.SparkSession = (sql.SparkSession.builder
	.appName("RailwaySimulationGenerator")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

## Data extraction

### Tracks

In [4]:
tracks_df = spark.read.csv(filenames["tracks_csv"], header=True, inferSchema=True, sep=";")
tracks_dict = dict()
schema : T.DataType = T.ArrayType(T.ArrayType(T.DoubleType()))
tracks_df = tracks_df.withColumn(
	"Path",
	F.from_json(F.col("Path"), schema)
)
for row in tracks_df.collect() :
	track_id = int(row["ID"])
	start = row["Departure_switch"]
	end = row["Arrival_switch"]
	distance = round(row["Length_m"], 6) 
	if track_id not in tracks_dict :
		tracks_dict[track_id] = {}
	tracks_dict[track_id] = {
		"start" : start,
		"end" : end,
		"distance" : distance
	}

In [5]:
for i in range(5) :
	track_id = rd.choice(list(tracks_dict.keys()))
	print(track_id, tracks_dict[track_id])

5433 {'start': 5789, 'end': 5602, 'distance': 66.365631}
2586 {'start': 2839, 'end': 2422, 'distance': 516.757091}
4457 {'start': 4726, 'end': 4852, 'distance': 564.540076}
18213 {'start': 14637, 'end': 14672, 'distance': 37.285115}
14209 {'start': 13370, 'end': 36, 'distance': 314.243084}


In [6]:
tracks = ET.parse(filenames["tracks_xml"]).getroot()
edges = tracks.findall("edge")
edges_dict = dict()

for edge in edges:
	edge_id = int(edge.get("id"))
	from_node = int(edge.get("from"))
	to_node = int(edge.get("to"))
	for track in tracks_dict : 
		if (tracks_dict[track]["start"] == from_node and tracks_dict[track]["end"] == to_node) :
			edges_dict[edge_id] = {
				"track_id" : track,
				"from" : from_node,
				"to" : to_node,
				"length" : tracks_dict[track]["distance"]
			}
			break
		if tracks_dict[track]["start"] == to_node and tracks_dict[track]["end"] == from_node :
			edges_dict[edge_id] = {
				"track_id" : track,
				"from" : to_node,
				"to" : from_node,
				"length" : tracks_dict[track]["distance"]
			}
			break

In [7]:
for i in range(5) :
	edge_id = rd.choice(list(edges_dict.keys()))
	print(edge_id, edges_dict[edge_id])

20099 {'track_id': 10049, 'from': 10177, 'to': 10178, 'length': 83.266847}
33271 {'track_id': 16635, 'from': 15788, 'to': 15787, 'length': 40.289867}
51464 {'track_id': 25732, 'from': 21540, 'to': 21560, 'length': 24.829704}
3637 {'track_id': 1818, 'from': 2054, 'to': 2055, 'length': 2423.297919}
38728 {'track_id': 19364, 'from': 16204, 'to': 17503, 'length': 1.005195}


### Station platforms 

In [8]:
plaforms_df = spark.read.csv(filenames["platforms_csv"], header=True, inferSchema=True, sep=",")
platforms_dict = dict()
for row in plaforms_df.collect() :
	station_id = int(row["Station_ID"])
	track_id = row["track_id"]
	city = row["Station_name"]
	if station_id not in platforms_dict :
		platforms_dict[station_id] = {
			"name" : city,
			"platforms" : []
		}
	platforms_dict[station_id]["platforms"].append(track_id)

In [9]:
for i in range(5) :
	platform = rd.choice(list(platforms_dict.keys()))
	print(platform, platforms_dict[platform])

105 {'name': 'BALEGEM-DORP', 'platforms': [1900, 10500]}
227 {'name': 'BRUXELLES-SCHUMAN', 'platforms': [16934, 17534, 11930, 16933]}
227 {'name': 'BRUXELLES-SCHUMAN', 'platforms': [16934, 17534, 11930, 16933]}
1079 {'name': 'RHODE-SAINT-GENESE', 'platforms': [1235, 1295]}
107 {'name': 'BALEN', 'platforms': [13854]}


In [10]:
trainstop_dict = dict()

# for edge_id in edges_dict :
# 	track_id = edges_dict[edge_id]["track_id"]
# 	station_id = None
# 	for station in platforms_dict :
# 		if track_id in platforms_dict[station]["platforms"] :
# 			station_id = station
# 			break
# 	if station_id is not None :
# 		if station_id not in trainstop_dict :
# 			trainstop_dict[station_id] = {
# 				"name" : platforms_dict[station_id]["name"],
# 				"platforms" : []
# 			}
# 		trainstop_dict[station_id]["platforms"].append(edge_id)

for station_id in platforms_dict :
	if station_id not in trainstop_dict :
		trainstop_dict[station_id] = {
			"name" : platforms_dict[station_id]["name"],
			"platforms" : []
		}
	for edge_id in edges_dict :
		track_id = edges_dict[edge_id]["track_id"]
		if track_id in platforms_dict[station_id]["platforms"] :
			trainstop_dict[station_id]["platforms"].append(edge_id)

In [11]:
for i in range(5) :
	station_id = rd.choice(list(trainstop_dict.keys()))
	print(station_id, trainstop_dict[station_id])

1018 {'name': 'ROUX', 'platforms': [10466, 10467, 10468, 10469, 10494, 10495, 29092, 29093, 29678, 29679]}
1147 {'name': 'TILFF', 'platforms': [11574, 11575, 11590, 11591]}
1256 {'name': 'ZANDBERGEN', 'platforms': [12090, 12091, 20912, 20913]}
1272 {'name': 'ZINGEM', 'platforms': [19052, 19053, 20100, 20101]}
335 {'name': 'DRONGEN', 'platforms': [868, 869, 7062, 7063, 7064, 7065, 44808, 44809]}


### Punctuality Data

In [12]:
punctuality_data_df = spark.read.csv(filenames["punctuality"], header=True, inferSchema=True, sep=";")
window : sql.Window = (
	sql.Window.partitionBy("TRAIN_NO","REAL_DATE_DEP") 
	.orderBy("PLANNED_DATETIME_DEP")
)
punctuality_data_df = (
	punctuality_data_df
	.withColumn(
		"NEXT_STOPPING_PLACE_ID",
		F.lead("STOPPING_PLACE_ID").over(window)
	)
)
punctuality_data_df = (punctuality_data_df.filter(
	(F.col("PLANNED_DATETIME_DEP") >= F.lit(START_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))) & 
	(F.col("PLANNED_DATETIME_DEP") <= F.lit(END_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))))
	.orderBy("TRAIN_NO", "REAL_DATE_DEP")
)

In [13]:
punctuality_data_df.show(5, truncate=False)

+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|RELATION_DIRECTION                      |LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|10      |ICE     |SNCB/NMBS |825              |37         |243      |243      |ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID|37         |01-01-2025   |01-01-2025   |2025-01-01 20:28:00 |2025-01-01 20:28:00 |266                   |
|10      |ICE     |SNCB/NMBS |266              |37         |297      |297      |ICE: FRANKFU

## Station to station

In [14]:
station_to_station_df = spark.read.csv(filenames["station_to_station"], header=True, inferSchema=True, sep=";")
punctuality_data_df.show(5, truncate=False)

+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|RELATION_DIRECTION                      |LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|10      |ICE     |SNCB/NMBS |825              |37         |243      |243      |ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID|37         |01-01-2025   |01-01-2025   |2025-01-01 20:28:00 |2025-01-01 20:28:00 |266                   |
|10      |ICE     |SNCB/NMBS |266              |37         |297      |297      |ICE: FRANKFU

In [15]:
network = dict()
for row in station_to_station_df.collect():
    station_a = row["Departure_station_id"]
    station_b = row["Arrival_station_id"]
    if station_a not in network:
        network[station_a] = []
    network[station_a].append(station_b)

In [16]:
for i in range(5) :
	station_id = rd.choice(list(network.keys()))
	print(station_id, network[station_id])

762 [936]
819 [1084, 58, 151, 1278, 30]
488 [243, 328]
752 [786, 723]
169 [958, 519]


## Trip generation

### Retrieving trips from punctuality data

In [17]:
punctuality_data_df = punctuality_data_df.toPandas()

trips = {}

for train_id, group in punctuality_data_df.groupby("RELATION"):
	stops = group["STOPPING_PLACE_ID"].tolist()
	# next_stops = group["NEXT_STOPPING_PLACE_ID"].tolist()

	path = []
	for s in stops :
		if s in path :
			break
		path.append(s) 
	trips[train_id] = path

In [18]:
punctuality_data_df.groupby("TRAIN_NO").head(20)

,TRAIN_NO,RELATION,TRAIN_SERV,STOPPING_PLACE_ID,LINE_NO_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,REAL_DATE_ARR,REAL_DATE_DEP,PLANNED_DATETIME_ARR,PLANNED_DATETIME_DEP,NEXT_STOPPING_PLACE_ID
0,10,ICE,SNCB/NMBS,825,37,243.0,243,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:28:00,2025-01-01 20:28:00,266.0
1,10,ICE,SNCB/NMBS,266,37,297.0,297,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,3,01-01-2025,01-01-2025,2025-01-01 20:39:00,2025-01-01 20:39:00,27.0
2,10,ICE,SNCB/NMBS,27,37,276.0,276,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:40:00,2025-01-01 20:40:00,726.0
3,10,ICE,SNCB/NMBS,726,36,223.0,127,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:43:00,2025-01-01 20:46:00,31.0
4,10,ICE,SNCB/NMBS,31,2,62.0,62,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,36,01-01-2025,01-01-2025,2025-01-01 20:51:00,2025-01-01 20:51:00,715.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267076,19976,IC 19-2,SNCB/NMBS,1154,94,NaN,3,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,04-01-2025,NaT,2025-01-04 21:44:00,427.0
267077,19976,IC 19-2,SNCB/NMBS,427,94,-32.0,-25,IC 19-2: TOURNAI -> LILLE FLANDRES,94,04-01-2025,04-01-2025,2025-01-04 21:48:00,2025-01-04 21:49:00,NaN
267078,19976,IC 19-2,SNCB/NMBS,1154,94,NaN,11,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,05-01-2025,NaT,2025-01-05 21:44:00,427.0
267079,19976,IC 19-2,SNCB/NMBS,427,94,-22.0,-15,IC 19-2: TOURNAI -> LILLE FLANDRES,94,05-01-2025,05-01-2025,2025-01-05 21:48:00,2025-01-05 21:49:00,NaN


In [19]:
final_trips  = {}
for trip in trips :
	path = []
	for s in trips[trip] :
		path.append(s) if s in platforms_dict else None
	if len(path) > 1 :
		final_trips[trip] = path

In [20]:
for train_id in final_trips :
	path = []
	for s in final_trips[train_id] :
		# path.append(platforms_dict[s]["name"]) if s in platforms_dict else None
		path.append((s, platforms_dict[s]["name"], s in platforms_dict))
	print(train_id, path)

EURST [(504, 'HALLE', True), (415, 'FOREST-MIDI', True), (220, 'BRUXELLES-MIDI', True), (217, 'BRUXELLES-CHAPELLE', True), (215, 'BRUXELLES-CENTRAL', True), (216, 'BRUXELLES-CONGRES', True), (221, 'BRUXELLES-NORD', True), (1048, 'SCHAERBEEK', True), (810, 'MECHELEN', True), (811, 'MECHELEN-NEKKERSPOEL', True), (1083, 'SINT-KATELIJNE-WAVER', True), (336, 'DUFFEL', True), (644, 'KONTICH-LINT', True), (590, 'HOVE', True), (866, 'MORTSEL-OUDE GOD', True), (863, 'MORTSEL', True), (139, 'ANTWERPEN-BERCHEM', True), (37, 'ANTWERPEN-CENTRAAL', True), (764, 'ANTWERPEN-LUCHTBAL', True), (1839, 'NOORDERKEMPEN', True)]
EXTRA [(977, 'PUURS', True), (1017, 'RUISBROEK-SAUVEGARDE', True), (188, 'BOOM', True), (905, 'NIEL', True), (1066, 'SCHELLE', True), (546, 'HEMIKSEM', True), (570, 'HOBOKEN-POLDER', True)]
IC 02 [(455, 'GENT-SINT-PIETERS', True), (447, 'GENTBRUGGE', True), (449, 'GENT-DAMPOORT', True), (130, 'BEERVELDE', True), (748, 'LOKEREN', True), (1073, 'SINAAI', True), (138, 'BELSELE', True), 

### Creating network as a graph

In [21]:
temp = {}
for edge_id in edges_dict:
	start, end = edges_dict[edge_id]["from"], edges_dict[edge_id]["to"]
	if start not in temp :
		temp[start] = set()
	temp[start].add(edge_id)
	if end not in temp :
		temp[end] = set()
	temp[end].add(edge_id)

In [22]:
import networkx as nx

G = nx.Graph()

for node in temp :
	tracks = list(temp[node])
	for i in range(len(tracks)) :
		for j in range(len(tracks)) : 
			if i != j :
				a = tracks[i]
				b = tracks[j]
				G.add_edge(a, b, track_id=f"{a} - {b}", weight=edges_dict[a]["length"])

### Creating paths between stations

In [23]:
def find_path_between_stations(G, station_A, station_B, station_A_tracks, station_B_tracks, station_to_station_paths):
	paths = None
	if (station_A, station_B) not in station_to_station_paths :
		station_to_station_paths[(station_A, station_B)] = dict()
		for platform_A in station_A_tracks :
			for platform_B in station_B_tracks :
				if (platform_A, platform_B) not in station_to_station_paths[(station_A, station_B)] :
					# print(f"searching path between {platform_A} and {platform_B}")
					try:
						path = nx.shortest_path(G, platform_A, platform_B, weight="weight")
						if len(path) == 0 :
							path = None
						else :
							length = nx.path_weight(G, path, weight="weight")
						if path is not None :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = (path, length)
						else :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = None
					except:
						# print("error")
						continue

In [25]:
station_to_station_paths = dict()
for station_id in network :
	for neighbor in network[station_id] :
		if station_id in trainstop_dict and neighbor in trainstop_dict :
			print(f"Finding path between {trainstop_dict[station_id]['name']} and {trainstop_dict[neighbor]['name']}")
			tracks_A = trainstop_dict[station_id]["platforms"]
			tracks_B = trainstop_dict[neighbor]["platforms"]
			find_path_between_stations(G, station_id, neighbor, tracks_A, tracks_B, station_to_station_paths)
			find_path_between_stations(G, neighbor, station_id, tracks_B, tracks_A, station_to_station_paths)

Finding path between MOLLEM and MERCHTEM
Finding path between MOLLEM and ASSE
Finding path between CALLENELLE and PERUWELZ
Finding path between CALLENELLE and MAUBRAY
Finding path between GENVAL and LA HULPE
Finding path between GENVAL and RIXENSART
Finding path between MAFFLE and ATH
Finding path between MAFFLE and MEVERGNIES-ATTRE
Finding path between STATTE and HUY
Finding path between STATTE and BAS-OHA
Finding path between BILZEN and TONGEREN
Finding path between BILZEN and BOKRIJK
Finding path between BILZEN and DIEPENBEEK
Finding path between ARCADES and DELTA
Finding path between ARCADES and BOONDAEL
Finding path between ARCADES and ETTERBEEK
Finding path between BEVERLO and LEOPOLDSBURG
Finding path between BEVERLO and BERINGEN
Finding path between NESSONVAUX and FRAIPONT
Finding path between NESSONVAUX and PEPINSTER
Finding path between DELTA and ARCADES
Finding path between DELTA and ETTERBEEK
Finding path between DELTA and MERODE
Finding path between DELTA and WATERMAEL
Fin

In [ ]:
for trip in final_trips :
	path = final_trips[trip]
	for i in range(len(path) - 1):
		if (path[i], path[i + 1]) not in station_to_station_paths :
			print(f"Finding path between {trainstop_dict[path[i]]['name']} and {trainstop_dict[path[i + 1]]['name']}")
			tracks_A = trainstop_dict[path[i]]["platforms"]
			tracks_B = trainstop_dict[path[i + 1]]["platforms"]
			find_path_between_stations(G, path[i], path[i + 1], tracks_A, tracks_B, station_to_station_paths)
		if (path[i + 1], path[i]) not in station_to_station_paths : 
			tracks_A = trainstop_dict[path[i + 1]]["platforms"]
			tracks_B = trainstop_dict[path[i]]["platforms"]
			find_path_between_stations(G, path[i + 1], path[i], tracks_A, tracks_B, station_to_station_paths)

Finding path between MORTSEL-OUDE GOD and MORTSEL
Finding path between MORTSEL and MORTSEL-OUDE GOD
Finding path between ALKEN and BLANKENBERGE
Finding path between BLANKENBERGE and ALKEN
Finding path between HARELBEKE and BISSEGEM
Finding path between BISSEGEM and HARELBEKE
Finding path between BRUXELLES-NORD and BINCHE
Finding path between BINCHE and BRUXELLES-NORD
Finding path between ASSESSE and NATOYE
Finding path between NATOYE and ASSESSE
Finding path between JEMEPPE-SUR-SAMBRE and MOUSTIER
Finding path between MOUSTIER and JEMEPPE-SUR-SAMBRE
Finding path between MILMORT and MONS
Finding path between MONS and MILMORT
Finding path between BERTRIX and NAMUR
Finding path between NAMUR and BERTRIX
Finding path between HERENT and BRAINE-LE-COMTE
Finding path between BRAINE-LE-COMTE and HERENT
Finding path between HERGENRATH and LIEGE-GUILLEMINS
Finding path between LIEGE-GUILLEMINS and HERGENRATH
Finding path between TROOZ and HASSELT
Finding path between HASSELT and TROOZ
Finding pa